# 🎙 Vlastní hlas pro Překladač (Piper) – trénování zdarma v Google Colab

Z vašich nahrávek (ZIP z **Překladače → Nastavení → Součásti → Vlastní hlas → Nahrávací studio → Export pro trénování**)
se dotrénuje český hlas **Piper** z hlasu Jirka. Výsledek (`.onnx` a `.onnx.json`) pak v Překladači importujete
a hlas čte knihy i audioknihy **bez internetu**.

**Než začnete:** nahoře *Runtime / Běhové prostředí → Změnit typ běhového prostředí → GPU (T4)*.
Pak spouštějte buňky postupně (▶). Trénování trvá dlouho (u hodiny nahrávek řádově desítky hodin na bezplatné grafice T4) – bezplatný Colab
se po několika hodinách odpojí, průběh je ale na vašem Google Disku a buňkami 1–5 se pokračuje dál.
Rychlejší je Colab Pro nebo počítač s grafikou NVIDIA. Poslechnout si mezivýsledek jde kdykoli buňkou 6.

> Tento sešit používá oficiální trénování Piper (projekt *OHF-Voice/piper1-gpl*). Když se něco změní v jejich
> instalaci, chybu napíše buňka 2 – pošlete ji a sešit se upraví.

In [16]:
# 1) Google Disk – sem se ukládá průběh trénování a hotový hlas
from google.colab import drive
drive.mount('/content/drive')
import os
WORK = '/content/drive/MyDrive/Prekladac-vlastni-hlas'
os.makedirs(WORK, exist_ok=True)
print('Pracovní složka:', WORK)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Pracovní složka: /content/drive/MyDrive/Prekladac-vlastni-hlas


In [1]:
# 2) Instalace trénování Piper (asi 5 minut)
!apt-get -qq install -y espeak-ng build-essential cmake ninja-build > /dev/null
!git clone -q https://github.com/OHF-Voice/piper1-gpl.git /content/piper1-gpl || true
%cd /content/piper1-gpl
!pip -q install -e '.[train]'
!./build_monotonic_align.sh
!pip -q install scikit-build && python3 setup.py build_ext --inplace > /content/build_ext.log 2>&1 || true
%cd /content
# espeakbridge (převod textu na hlásky): když se ho nepodaří sestavit, vezme se hotový z oficiálního balíčku Piper
import glob, os, re, shutil, subprocess, sys, zipfile
SRC = '/content/piper1-gpl/src/piper'
if not glob.glob(SRC + '/espeakbridge*.so'):
    ver = re.search(r'version="([^"]+)"', open('/content/piper1-gpl/setup.py', encoding='utf-8').read())
    pkg = 'piper-tts==' + ver.group(1) if ver else 'piper-tts'
    subprocess.run([sys.executable, '-m', 'pip', 'download', pkg, '--no-deps', '-q', '-d', '/content/wheel'], check=True)
    z = zipfile.ZipFile(glob.glob('/content/wheel/piper_tts-*.whl')[0])
    for n in z.namelist():
        if n.startswith('piper/espeakbridge') or n.startswith('piper/espeak-ng-data/'):
            z.extract(n, '/content/whl_x')
    shutil.copy(glob.glob('/content/whl_x/piper/espeakbridge*.so')[0], SRC)
    shutil.rmtree(SRC + '/espeak-ng-data', ignore_errors=True)
    shutil.copytree('/content/whl_x/piper/espeak-ng-data', SRC + '/espeak-ng-data')
sys.path.insert(0, '/content/piper1-gpl/src')
from piper.phonemize_espeak import EspeakPhonemizer
print('✓ Instalace hotová, převod na hlásky funguje:', EspeakPhonemizer().phonemize('cs', 'Dobrý den.'))

fatal: destination path '/content/piper1-gpl' already exists and is not an empty directory.
/content/piper1-gpl
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for piper-tts (pyproject.toml) ... done
Compiling /content/piper1-gpl/src/piper/train/vits/monotonic_align/core.pyx because it changed.
[1/1] Cythonizing /content/piper1-gpl/src/piper/train/vits/monotonic_align/core.pyx
/usr/local/lib/python3.13/dist-packages/Cython/Compiler/Main.py:381: FutureWarning: Cython directive 'language_level' not set, using '3str' for now (Py3). This has changed from earlier releases! File: /content/piper1-gpl/src/piper/train/vits/monotonic_align/core.pyx
  tree = Parsing.p_module(s, pxd, full_module_name)
performance hint: core.pyx:7:5: Exception check on 'maximum_path_each' will always require the GIL to be acquired.
Possib

In [2]:
# 3) ZIP(y) s nahrávkami z Překladače – nejspolehlivěji přes Google Disk:
#    drive.google.com → Můj disk → Prekladac-vlastni-hlas → přetáhnout ZIP (i několik najednou)
import os
if 'WORK' not in globals():  # po restartu běhového prostředí
    from google.colab import drive
    drive.mount('/content/drive')
    WORK = '/content/drive/MyDrive/Prekladac-vlastni-hlas'
import glob, zipfile, os, shutil
zips = glob.glob(WORK + '/*.zip')
if not zips:
    try:
        from google.colab import files
        up = files.upload()  # okno pro nahrání (při blokovaných cookies nefunguje)
        for name, data in up.items():
            open(os.path.join(WORK, name), 'wb').write(data)
    except Exception as e:
        print('⚠ Okno pro nahrání se v prohlížeči nespustilo (' + type(e).__name__ + ').')
        print('  Nahrajte ZIP na https://drive.google.com do složky Prekladac-vlastni-hlas a spusťte buňku znovu.')
    zips = glob.glob(WORK + '/*.zip')
assert zips, 'Ve složce Prekladac-vlastni-hlas na Google Disku zatím není žádný ZIP.'

# všechny ZIPy se spojí do jednoho vstupu (stejné věty se nezdvojí)
shutil.rmtree('/content/data', ignore_errors=True)
DATA = '/content/data/hlas'
os.makedirs(DATA + '/wavs', exist_ok=True)
lines, seen, names = [], set(), []
for k, z in enumerate(sorted(zips, key=os.path.getmtime)):
    tmp = f'/content/zip{k}'
    shutil.rmtree(tmp, ignore_errors=True)
    zipfile.ZipFile(z).extractall(tmp)
    for meta in glob.glob(tmp + '/*/metadata.csv'):
        base = os.path.dirname(meta)
        names.append(os.path.basename(base))
        for line in open(meta, encoding='utf-8'):
            wav, _, text = line.rstrip('\n').partition('|')
            if not text or text.lower() in seen or not os.path.isfile(f'{base}/wavs/{wav}'):
                continue
            seen.add(text.lower())
            new = f'{len(lines) + 1:05d}.wav'
            shutil.copy(f'{base}/wavs/{wav}', f'{DATA}/wavs/{new}')
            lines.append(f'{new}|{text}')
open(DATA + '/metadata.csv', 'w', encoding='utf-8').write('\n'.join(lines) + '\n')
VOICE = names[-1] if names else 'muj_hlas'
print(f'ZIPů: {len(zips)} · hlas: {VOICE} · vět celkem: {len(lines)}')
if len(lines) < 300: print('⚠ Méně než 300 vět – hlas bude méně přesný; nahrajte víc, pokud to jde.')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
ZIPů: 1 · hlas: michal_mikrofon · vět celkem: 544


In [3]:
# 4) Výchozí hlas Jirka – stáhne se jen jednou a uloží na Google Disk
import os, glob, shutil
CKPT_START = WORK + '/jirka-start.ckpt'
PATH = 'cs/cs_CZ/jirka/medium/epoch=2099-step=92400.ckpt'
if os.path.isfile(CKPT_START):
    print('Jirka už je na Disku.')
else:
    local = glob.glob('/content/start/**/*.ckpt', recursive=True)
    if not local:
        from huggingface_hub import hf_hub_download
        for typ in ('model', 'dataset'):
            try:
                print(f'Stahuji Jirku (849 MB, typ {typ})…')
                local = [hf_hub_download('rhasspy/piper-checkpoints', PATH, repo_type=typ,
                                         local_dir='/content/start', token=False)]
                break
            except Exception as e:
                print('  nejde:', type(e).__name__, str(e)[:200])
    assert local, 'Jirku se nepodařilo stáhnout – pošlete mi celý tento výpis.'
    print('Staženo:', local[0], round(os.path.getsize(local[0]) / 1e6), 'MB')
    print('Kopíruji na Google Disk (může trvat několik minut)…')
    shutil.copy(local[0], CKPT_START + '.part')
    os.replace(CKPT_START + '.part', CKPT_START)
print('✓ Výchozí hlas:', CKPT_START, round(os.path.getsize(CKPT_START) / 1e6), 'MB')

Jirka už je na Disku.
✓ Výchozí hlas: /content/drive/MyDrive/Prekladac-vlastni-hlas/jirka-start.ckpt 849 MB


In [6]:
# 5) Trénování – výpis se ukládá do Prekladac-vlastni-hlas/trenink.log; při chybě se ukáže jeho konec.
!nvidia-smi
import torch
print('torch:', torch.__version__, '· CUDA v torchi:', torch.version.cuda, '· GPU vidí:', torch.cuda.is_available())
import os
if 'WORK' not in globals():  # po restartu běhového prostředí
    from google.colab import drive
    drive.mount('/content/drive')
    WORK = '/content/drive/MyDrive/Prekladac-vlastni-hlas'
#    Po odpojení stačí znovu spustit buňky 1–5 (pokračuje se od last.ckpt).
import glob, subprocess, torch
assert torch.cuda.is_available(), ('Běží bez grafické karty – trénink by trval dny a spadl by na RAM. '
    'Běhové prostředí → Změnit typ běhového prostředí → T4 GPU, pak znovu buňky 1–5.')
print('GPU:', torch.cuda.get_device_name(0))
assert os.path.isfile(CKPT_START) or glob.glob(WORK + '/lightning_logs/*/checkpoints/last.ckpt'), \
    'Nejdřív nechte doběhnout buňku 4 (✓ Výchozí hlas … 849 MB).'
last = sorted(glob.glob(WORK + '/lightning_logs/*/checkpoints/last.ckpt'), key=os.path.getmtime)
START = ['--ckpt_path', last[-1]] if last else ['--model.warmstart_ckpt', CKPT_START]
print('Pokračuji z', last[-1]) if last else print('Začínám z hlasu', CKPT_START)
EPOCHS = 2000   # doporučení projektu Piper pro dotrénování z méně než ~5 h nahrávek
cmd = ['python3', '-m', 'piper.train', 'fit',
       '--data.voice_name', VOICE, '--data.csv_path', f'{DATA}/metadata.csv', '--data.audio_dir', f'{DATA}/wavs',
       '--model.sample_rate', '22050', '--data.espeak_voice', 'cs', '--data.cache_dir', '/content/cache',
       '--data.config_path', f'{WORK}/{VOICE}.onnx.json', '--data.batch_size', '16',
       '--trainer.max_epochs', str(EPOCHS), '--trainer.default_root_dir', WORK] + START
log = open(f'{WORK}/trenink.log', 'w', encoding='utf-8')
p = subprocess.Popen(cmd, cwd=WORK, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in p.stdout:
    log.write(line); log.flush()
    if 'Epoch' in line or 'Error' in line or 'error' in line:
        print(line.rstrip()[-200:])
p.wait(); log.close()
if p.returncode:
    print('\n❌ Trénování skončilo chybou. Posledních 40 řádků výpisu:\n')
    print(''.join(open(f'{WORK}/trenink.log', encoding='utf-8').readlines()[-40:]))

/bin/bash: line 1: nvidia-smi: command not found
torch: 2.14.1+cpu · CUDA v torchi: None · GPU vidí: False


AssertionError: Běží bez grafické karty – trénink by trval dny a spadl by na RAM. Běhové prostředí → Změnit typ běhového prostředí → T4 GPU, pak znovu buňky 1–5.

In [ ]:
# 6) Export hlasu (.onnx + .onnx.json) a krátká ukázka
import glob, json
# nejnovější last.ckpt (projekt Piper doporučuje začít s ním; porovnat jde i val_mos / val_mel)
ck = sorted(glob.glob(WORK + '/lightning_logs/*/checkpoints/last.ckpt'), key=os.path.getmtime)[-1]
!python3 -m piper.train.export_onnx --checkpoint "{ck}" --output-file "{WORK}/{VOICE}.onnx"
cfg = json.load(open(f'{WORK}/{VOICE}.onnx.json', encoding='utf-8'))
cfg['dataset'] = VOICE
json.dump(cfg, open(f'{WORK}/{VOICE}.onnx.json', 'w', encoding='utf-8'), ensure_ascii=False)
!pip -q install piper-tts
!echo "Dobrý den, tohle je můj vlastní hlas v Překladači." | piper -m "{WORK}/{VOICE}.onnx" -f /content/ukazka.wav
from IPython.display import Audio
Audio('/content/ukazka.wav')

In [ ]:
# 7) Stažení – oba soubory dejte do jedné složky a v Překladači: Nastavení → Součásti → Importovat hlas (.onnx)
from google.colab import files
files.download(f'{WORK}/{VOICE}.onnx')
files.download(f'{WORK}/{VOICE}.onnx.json')